# Notebook 1
# Préparation et nettoyage des données


4 étapes simples : charger > extraire > créer des indicateurs > nettoyer.


## 1 - Chargement des fichiers FAO

In [13]:
import pandas as pd
import numpy as np

df_food = pd.read_csv('DisponibiliteAlimentaire_2017.csv')
df_pop  = pd.read_csv('Population_2000_2018.csv')

print('Bilans alimentaires :', df_food.shape[0], 'lignes', df_food.shape[1], 'colonnes')
print('Population          :', df_pop.shape[0], 'lignes', df_pop.shape[1], 'colonnes')

Bilans alimentaires : 176600 lignes 14 colonnes
Population          : 4411 lignes 15 colonnes


## 2 - Population 2017

On isole l'année 2017 pour connaître la taille de chaque marché

La colonne s'appelle `Population_1000h` : la population est exprimée en **milliers d'habitants** (ex. 67 000 = 67 millions, la France)

In [14]:
pop = df_pop[df_pop['Année'] == 2017][['Zone', 'Valeur']].copy()
pop.columns = ['Zone', 'Population_1000h']
pop['Population_1000h'] = pd.to_numeric(pop['Population_1000h'], errors='coerce')

print(pop.shape[0], 'pays avec population 2017')
pop.head()

236 pays avec population 2017


,Zone,Population_1000h
17,Afghanistan,36296.113
36,Afrique du Sud,57009.756
55,Albanie,2884.169
74,Algérie,41389.189
93,Allemagne,82658.409


## 3 - Extraction des variables sur la volaille

On extrait les données pour le produit Viande de Volailles, en 6 éléments :

`Volaille_kg_hab` = Dispo. alimentaire (kg/pers/an) = la consommation 
`Volaille_kcal_jour`= Dispo. (Kcal/pers/jour) = l'intensité calorique 
`Volaille_prot_g_jour`= Dispo. protéines (g/pers/jour) = l'apport protéique 
`Volaille_Production_kt`= Production = la production locale 
`Volaille_Import_kt`= Importations - Quantité = ce qu'on importe 
`Volaille_Export_kt`= Exportations - Quantité = ce qu'on exporte
`Volaille_DI_kt`= Disponibilité intérieure = le marché total 

In [15]:
print(df_food["Produit"].unique())
print(df_food["Élément"].unique())

['Blé et produits' 'Riz et produits' 'Orge et produits' 'Maïs et produits'
 'Seigle et produits' 'Avoine' 'Millet et produits' 'Sorgho et produits'
 'Céréales, Autres' 'Pommes de Terre et produits' 'Ignames' 'Racines nda'
 'Sucre, canne' 'Sucre, betterave' 'Sucre Eq Brut' 'Edulcorants Autres'
 'Miel' 'Haricots' 'Pois' 'Légumineuses Autres et produits'
 'Noix et produits' 'Soja' 'Arachides Decortiquees' 'Graines de tournesol'
 'Graines Colza/Moutarde' 'Graines de coton' 'Coco (Incl Coprah)' 'Sésame'
 'Olives' 'Plantes Oleiferes, Autre' 'Huile de Soja' "Huile d'Arachide"
 'Huile de Tournesol' 'Huile de Colza&Moutarde' 'Huile Graines de Coton'
 'Huile de Palmistes' 'Huile de Palme' 'Huile de Coco' 'Huile de Sésame'
 "Huile d'Olive" 'Huile de Son de Riz' 'Huile de Germe de Maïs'
 'Huil Plantes Oleif Autr' 'Tomates et produits' 'Oignons'
 'Légumes, Autres' 'Oranges, Mandarines' 'Citrons & Limes et produits'
 'Pamplemousse et produits' 'Agrumes, Autres' 'Bananes'
 'Pommes et produits' 'Anana

In [16]:
df_food.head()

,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole
0,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5511,Production,2511,Blé et produits,2017,2017,Milliers de tonnes,4281.0,S,Données standardisées
1,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5611,Importations - Quantité,2511,Blé et produits,2017,2017,Milliers de tonnes,2302.0,S,Données standardisées
2,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5072,Variation de stock,2511,Blé et produits,2017,2017,Milliers de tonnes,-119.0,S,Données standardisées
3,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5911,Exportations - Quantité,2511,Blé et produits,2017,2017,Milliers de tonnes,0.0,S,Données standardisées
4,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5301,Disponibilité intérieure,2511,Blé et produits,2017,2017,Milliers de tonnes,6701.0,S,Données standardisées


In [17]:
def extrait(df, produit, element, nom_colonne):
    tmp = df[(df['Produit'] == produit) & (df['Élément'] == element)][['Zone', 'Valeur']].copy()
    tmp['Valeur'] = pd.to_numeric(tmp['Valeur'], errors='coerce')
    tmp.columns = ['Zone', nom_colonne]
    return tmp

# 7 variables exta
conso_kg   = extrait(df_food, 'Viande de Volailles',
                     'Disponibilité alimentaire en quantité (kg/personne/an)', 'Volaille_kg_hab')
conso_kcal = extrait(df_food, 'Viande de Volailles',
                     'Disponibilité alimentaire (Kcal/personne/jour)', 'Volaille_kcal_jour')
conso_prot = extrait(df_food, 'Viande de Volailles',
                     'Disponibilité de protéines en quantité (g/personne/jour)', 'Volaille_prot_g_jour')

production   = extrait(df_food, 'Viande de Volailles', 'Production', 'Volaille_Production_kt')
importations = extrait(df_food, 'Viande de Volailles', 'Importations - Quantité', 'Volaille_Import_kt')
exportations = extrait(df_food, 'Viande de Volailles', 'Exportations - Quantité', 'Volaille_Export_kt')
dispo_int    = extrait(df_food, 'Viande de Volailles', 'Disponibilité intérieure', 'Volaille_DI_kt')


## 4 - Viande totale (toutes les viandes)

Pour calculer la part du poulet dans l'assiette de viandes
=  6 cat de viande en kg et kcal

In [18]:
#liste de types de viandes 
viandes = ['Viande de Volailles', 'Viande de Bovins', 'Viande de Suides',
           "Viande d'Ovins/Caprins", 'Viande de Anim Aquatiq', 'Viande, Autre']
# def de variable somme_viande pour chaque pays
def somme_viande(element, nom_final):
    morceaux = []
    #loop for pour un df avec la conso de kg de viande (toutes réunies) par hab.
    for v in viandes:
        t = extrait(df_food, v, element, v)
        morceaux.append(t.set_index('Zone')[v])
    total = pd.concat(morceaux, axis=1)
    total[nom_final] = total.sum(axis=1, skipna=True)
    return total[[nom_final]].reset_index()

viande_kg   = somme_viande('Disponibilité alimentaire en quantité (kg/personne/an)',
                           'Viande_Tot_kg_hab')
viande_kcal = somme_viande('Disponibilité alimentaire (Kcal/personne/jour)',
                           'Viande_Tot_kcal_jour')

print(viande_kg.shape[0], 'pays avec viande totale calculée')
viande_kg.head()

172 pays avec viande totale calculée


,Zone,Viande_Tot_kg_hab
0,Afghanistan,8.59
1,Afrique du Sud,60.01
2,Albanie,47.51
3,Algérie,18.05
4,Allemagne,87.78


## 5 - Assemblage en un seul tableau

In [19]:
tables = [pop, conso_kg, conso_kcal, conso_prot,
          production, importations, exportations, dispo_int,
          viande_kg, viande_kcal]

df = pop.copy()
for t in tables[1:]:
    df = df.merge(t, on='Zone', how='outer')

print('Dataset assemblé :', df.shape[0], 'pays x', df.shape[1], 'colonnes')
df.head()

Dataset assemblé : 236 pays x 11 colonnes


,Zone,Population_1000h,Volaille_kg_hab,Volaille_kcal_jour,Volaille_prot_g_jour,Volaille_Production_kt,Volaille_Import_kt,Volaille_Export_kt,Volaille_DI_kt,Viande_Tot_kg_hab,Viande_Tot_kcal_jour
0,Afghanistan,36296.113,1.53,5.0,0.54,28.0,29.0,NaN,57.0,8.59,47.0
1,Afrique du Sud,57009.756,35.69,143.0,14.11,1667.0,514.0,63.0,2118.0,60.01,308.0
2,Albanie,2884.169,16.36,85.0,6.26,13.0,38.0,0.0,47.0,47.51,263.0
3,Algérie,41389.189,6.38,22.0,1.97,275.0,2.0,0.0,277.0,18.05,83.0
4,Allemagne,82658.409,19.47,71.0,7.96,1514.0,842.0,646.0,1739.0,87.78,360.0



## 6 - Création des indicateurs clés

Variable = Formule + Signification

Part_Poulet_pct = volaille / viande totale & Préférence pour le poulet  

Dependance_Import_pct = import / dispo. intérieure & Besoin d'importer  

Autosuffisance_pct = production / dispo. intérieure & Capacité locale  

Volaille_Consommation_Tot_kt` = kg/hab × population & Volume du marché   


Balance_Com_Volaille_kt = export − import & Position commerciale

In [24]:
df['Part_Poulet_Viande_pct']     = (df['Volaille_kg_hab'] / df['Viande_Tot_kg_hab']) * 100
df['Dependance_Import_pct']      = (df['Volaille_Import_kt'] / df['Volaille_DI_kt']) * 100
df['Autosuffisance_pct']         = (df['Volaille_Production_kt'] / df['Volaille_DI_kt']) * 100
df['Volaille_Consommation_Tot_kt'] = df['Volaille_kg_hab'] * df['Population_1000h'] / 1e6
df['Balance_Com_Volaille_kt']    = df['Volaille_Export_kt'] - df['Volaille_Import_kt']


## 7 - Filtrage des pays complets

2 filtres :
1. il faut au moins la **population** et la **consommation de volaille** connues ;
2. vérification : la population couverte doit dépasser **60 %** de la population mondiale.

In [21]:
df_propre = df.dropna(subset=['Population_1000h', 'Volaille_kg_hab']).copy()

pop_couverte = df_propre['Population_1000h'].sum()
pop_mondiale = 7_600_000   # milliers d'habitants, source ONU 2017

print('Pays retenus        :', df_propre.shape[0])
print('Population couverte :', f'{pop_couverte:,.0f} milliers')
print(f'Soit {pop_couverte/pop_mondiale*100:.1f} % de la population mondiale')

Pays retenus        : 172
Population couverte : 7,368,782 milliers
Soit 97.0 % de la population mondiale


## 8 - Imputation + sélection des 13 variables + sauvegarde

Valeurs manquantes = médiane (+ robuste que la moyenne pq  Chine ou les États-Unis ont chiffres très élévées).

On ne garde que les 13 variables et les zones; les autres colonnes ne sont plus utiles



In [22]:
# Imputation par la médiane sur toutes les colonnes numériques
for col in df_propre.select_dtypes(include='number').columns:
    df_propre[col] = df_propre[col].fillna(df_propre[col].median())

# Les 13 variables de l'analyse (identiques à l'annexe du rapport)
variables_13 = [
    'Population_1000h',             # taille du marché
    'Volaille_kg_hab',              # consommation per capita
    'Volaille_kcal_jour',           # intensité calorique
    'Volaille_prot_g_jour',         # apport protéique
    'Part_Poulet_Viande_pct',       # préférence pour le poulet
    'Dependance_Import_pct',        # ouverture aux importations
    'Autosuffisance_pct',           # capacité locale
    'Volaille_Consommation_Tot_kt', # volume du marché
    'Viande_Tot_kg_hab',            # consommation de viande totale
    'Viande_Tot_kcal_jour',         # kcal viande totale
    'Volaille_Production_kt',       # production locale
    'Volaille_Export_kt',           # capacité exportatrice
    'Balance_Com_Volaille_kt',      # balance commerciale
]

df_final = df_propre[['Zone'] + variables_13].copy()
df_final.to_csv('dataset_analyse.csv', index=False)

print('Dataset sauvegardé : dataset_analyse.csv')
print('Dimensions :', df_final.shape, '=172 pays x 13 variables + Zone')

Dataset sauvegardé : dataset_analyse.csv
Dimensions : (172, 14) =172 pays x 13 variables + Zone


## Aperçu des statistiques

Vérification finale avec les attentes écrite en haut et pas de valuers manquantes

In [23]:
df_final.describe().round(2)

,Population_1000h,Volaille_kg_hab,Volaille_kcal_jour,Volaille_prot_g_jour,Part_Poulet_Viande_pct,Dependance_Import_pct,Autosuffisance_pct,Volaille_Consommation_Tot_kt,Viande_Tot_kg_hab,Viande_Tot_kcal_jour,Volaille_Production_kt,Volaille_Export_kt,Balance_Com_Volaille_kt
count,172.00,172.00,172.00,172.00,172.00,172.00,172.00,172.00,172.00,172.00,172.00,172.00,172.00
mean,42841.75,20.21,74.56,7.12,40.49,45.56,78.25,0.65,48.74,235.34,709.95,104.40,17.31
std,153063.71,15.86,60.57,5.60,20.55,51.60,48.48,2.13,29.65,150.70,2474.01,457.92,465.10
min,52.04,0.13,0.00,0.04,1.70,0.00,0.00,0.00,3.78,14.00,0.00,0.00,-1059.00
25%,2874.48,6.44,22.00,2.16,23.41,4.31,40.00,0.03,21.19,107.25,14.75,0.00,-30.50
50%,9757.83,18.09,64.00,6.50,38.09,24.75,87.95,0.09,45.44,212.00,70.00,3.00,-10.00
75%,30138.74,30.04,105.50,10.36,55.43,80.21,100.00,0.35,71.46,339.75,368.75,12.00,-2.00
max,1421021.79,72.31,243.00,27.87,86.67,323.93,304.61,18.10,137.06,782.00,21914.00,4223.00,4220.00
